# 02 - Preprocessing

**Mục tiêu:** tạo dataset sạch để huấn luyện mô hình fraud detection.

Notebook này gọi pipeline trong `src/preprocessing.py` để đảm bảo preprocessing có thể chạy lại được, không phụ thuộc vào thao tác thủ công trong notebook.


## 1. Setup

Tự xác định thư mục gốc dự án và import module preprocessing từ `src`.


In [1]:
from pathlib import Path
import sys

import pandas as pd

try:
    PROJECT_ROOT = Path(__file__).resolve().parents[1]
except NameError:
    cwd = Path.cwd().resolve()
    PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from preprocessing import preprocess

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "reports" / "tables"

print("Project root:", PROJECT_ROOT)
print("Raw dir:", RAW_DIR)
print("Processed dir:", PROCESSED_DIR)


Project root: D:\main\Full\documents\uni\Nam 2\Ky 2\big data\cki\fraud-detection-bigdata
Raw dir: D:\main\Full\documents\uni\Nam 2\Ky 2\big data\cki\fraud-detection-bigdata\data\raw
Processed dir: D:\main\Full\documents\uni\Nam 2\Ky 2\big data\cki\fraud-detection-bigdata\data\processed


## 2. Preprocessing Plan

Pipeline thực hiện các bước:

- Đọc `fraudTrain.csv` và `fraudTest.csv`.
- Tạo feature thời gian: `transaction_hour`, `transaction_dayofweek`, `transaction_month`, `weekend_flag`.
- Tạo `customer_age` từ `dob` và thời điểm giao dịch.
- Tạo `customer_merchant_distance_km` từ tọa độ khách hàng và merchant.
- One-hot encode: `category`, `gender`, `state`.
- Frequency encode: `merchant`, `job`, `city`.
- Loại bỏ các cột định danh trực tiếp như `cc_num`, `first`, `last`, `street`, `trans_num`.
- Scale các feature số bằng `StandardScaler` fit trên train set.


## 3. Run Pipeline

Cell này sẽ sinh file processed trong `data/processed` và bảng tóm tắt trong `reports/tables`.


In [2]:
train_processed, test_processed = preprocess(
    raw_dir=RAW_DIR,
    processed_dir=PROCESSED_DIR,
    table_dir=TABLE_DIR,
)

print("Train processed shape:", train_processed.shape)
print("Test processed shape:", test_processed.shape)


Train processed shape: (1296675, 85)
Test processed shape: (555719, 85)


## 4. Validate Output

Kiểm tra nhanh missing values, target distribution và các feature sau preprocessing.


In [3]:
processed_files = sorted(PROCESSED_DIR.glob("*.csv"))
for path in processed_files:
    print(path.name, f"{path.stat().st_size / 1024**2:,.2f} MB")


fraud_test_processed.csv 208.57 MB
fraud_train_processed.csv 487.60 MB


In [4]:
validation = pd.DataFrame({
    "dataset": ["train", "test"],
    "rows": [len(train_processed), len(test_processed)],
    "columns": [train_processed.shape[1], test_processed.shape[1]],
    "missing_values": [train_processed.isna().sum().sum(), test_processed.isna().sum().sum()],
    "fraud_rate_percent": [
        train_processed["is_fraud"].mean() * 100,
        test_processed["is_fraud"].mean() * 100,
    ],
})
validation


,dataset,rows,columns,missing_values,fraud_rate_percent
0,train,1296675,85,0,0.578865
1,test,555719,85,0,0.385986


In [5]:
train_processed.head()


,amt,zip,lat,long,city_pop,unix_time,merch_lat,merch_long,transaction_hour,transaction_dayofweek,...,state_TN,state_TX,state_UT,state_VA,state_VT,state_WA,state_WI,state_WV,state_WY,is_fraud
0,-0.407826,28654,-0.484420,0.657620,-0.282589,-1.858664,-0.494354,0.593864,0,1,...,0,0,0,0,0,0,0,0,0,0
1,0.230039,99160,2.039120,-2.033870,-0.293670,-1.858662,2.078699,-2.030341,0,1,...,0,0,0,0,0,1,0,0,0,0
2,0.934149,83252,0.717754,-1.601537,-0.280406,-1.858662,0.902849,-1.592323,0,1,...,0,0,0,0,0,0,0,0,0,0
3,-0.158132,59632,1.515617,-1.590766,-0.287742,-1.858660,1.662886,-1.621848,0,1,...,0,0,0,0,0,0,0,0,0,0
4,-0.177094,24433,-0.023035,0.782279,-0.293835,-1.858651,0.026941,0.841909,0,1,...,0,0,0,1,0,0,0,0,0,0


In [6]:
train_processed.dtypes.value_counts()


int64      70
float64    12
int32       3
Name: count, dtype: int64

## 5. Output Files

Kết quả chính:

- `data/processed/fraud_train_processed.csv`
- `data/processed/fraud_test_processed.csv`
- `reports/tables/preprocessing_report.json`
- `reports/tables/processed_feature_summary.csv`
- `reports/tables/frequency_encoding_summary.csv`
- `reports/tables/scaler_summary.csv`

Bước tiếp theo là dùng processed data để làm Spark processing hoặc modeling.
